In [1]:
# import model_loader
# import pipeline
# from PIL import Image
# from pathlib import Path
# from transformers import CLIPTokenizer
# import torch

# DEVICE = "cpu"

# ALLOW_CUDA = False
# ALLOW_MPS = True

# if torch.cuda.is_available() and ALLOW_CUDA:
#     DEVICE = "cuda"
# elif (torch.has_mps or torch.backends.mps.is_available()) and ALLOW_MPS:
#     DEVICE = "mps"
# print(f"Using device: {DEVICE}")

# tokenizer = CLIPTokenizer("../data/vocab.json", merges_file="../data/merges.txt")
# model_file = "../data/v1-5-pruned-emaonly.ckpt"
# models = model_loader.preload_models_from_standard_weights(model_file, DEVICE)

# ## TEXT TO IMAGE

# prompt = (
#     "dashcam perspective, in driver seat of a car. realistic, detailed rainy weather, wet road."
# )

# uncond_prompt = (
#     "cartoon, illustration, painting, unrealistic, clear weather, sunny, blurry, low detail."
# )

# do_cfg = True
# cfg_scale = 11  # min: 1, max: 14

# ## IMAGE TO IMAGE

# input_image = None
# # # Comment to disable image to image
# image_path = "../images/dog.jpg"
# # # input_image = Image.open(image_path)
# # # Higher values means more noise will be added to the input image, so the result will further from the input image.
# # # Lower values means less noise is added to the input image, so output will be closer to the input image.
# strength = 0.9

# ## SAMPLER

# sampler = "ddpm"
# num_inference_steps = 50
# seed = 89

# output_image = pipeline.generate(
#     prompt=prompt,
#     uncond_prompt=uncond_prompt,
#     input_image=input_image,
#     strength=strength,
#     do_cfg=do_cfg,
#     cfg_scale=cfg_scale,
#     sampler_name=sampler,
#     n_inference_steps=num_inference_steps,
#     seed=seed,
#     models=models,
#     device=DEVICE,
#     idle_device="cpu",
#     tokenizer=tokenizer,
# )

# # Combine the input image and the output image into a single image.
# Image.fromarray(output_image)

In [2]:

# # Save the generated image to disk
# output_dir = Path("../outputs")
# output_dir.mkdir(parents=True, exist_ok=True)
# output_path = output_dir / f"generated_seed_{seed}.png"

# Image.fromarray(output_image).save(output_path)
# print(f"Saved image to {output_path}")

In [ ]:
import model_loader
import pipeline
from transformers import CLIPTokenizer
from PIL import Image
from pathlib import Path
import torch
import random

# -----------------------------
# Configuration
# -----------------------------
DEVICE = "cpu"
ALLOW_CUDA = False
ALLOW_MPS = True

NUM_IMAGES_PER_CLASS = 50
OUTPUT_DIR = Path("../synthetic_imagenet_semantic")

classes = {
    "elephant": {
        "positives": [
            "in a dry savannah with tall grass",
            "near a water source with reflections",
            "partially obscured by dust or foliage",
            "walking slowly across open terrain",
            "viewed from a low angle emphasizing scale",
            "side profile with visible tusks",
            "standing in a forest clearing",
            "background cluttered with vegetation",
        ],
        "negative": (
            "cartoon, illustration, fantasy, unrealistic anatomy, "
            "small animal, incorrect scale, zoo enclosure, "
            "studio lighting, artificial colors, text, watermark"
        ),
    },
    "deer": {
        "positives": [
            "in a dense forest with uneven lighting",
            "partially hidden behind trees",
            "crossing a clearing at dawn",
            "standing alert with head turned",
            "viewed at a distance with scale ambiguity",
            "low-contrast background blending with fur",
            "in mist or light fog",
            "mid-stride while walking",
        ],
        "negative": (
            "cartoon, illustration, fantasy, incorrect anatomy, "
            "predator features, antlers on incorrect species, "
            "studio lighting, artificial colors, text, watermark"
        ),
    },
    "tiger": {
        "positives": [
            "moving through tall grass",
            "partially obscured by shadows",
            "side view emphasizing stripes",
            "low lighting with high contrast",
            "walking through a forest environment",
            "viewed from behind foliage",
            "head-on view with intense gaze",
            "in a cluttered natural background",
        ],
        "negative": (
            "cartoon, illustration, fantasy, incorrect stripe patterns, "
            "domestic cat features, zoo enclosure, "
            "studio lighting, artificial colors, text, watermark"
        ),
    },
}

# -----------------------------
# Device selection
# -----------------------------
if torch.cuda.is_available() and ALLOW_CUDA:
    DEVICE = "cuda"
elif (torch.has_mps or torch.backends.mps.is_available()) and ALLOW_MPS:
    DEVICE = "mps"

print(f"Using device: {DEVICE}")

# -----------------------------
# Load models
# -----------------------------
tokenizer = CLIPTokenizer("../data/vocab.json", merges_file="../data/merges.txt")
model_file = "../data/v1-5-pruned-emaonly.ckpt"
models = model_loader.preload_models_from_standard_weights(model_file, DEVICE)

# -----------------------------
# Sampler + CFG settings
# -----------------------------
sampler = "ddpm"
num_inference_steps = 75

do_cfg = True
cfg_scale = 7.5

# -----------------------------
# Dataset generation
# -----------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for cls, cfg in classes.items():
    class_dir = OUTPUT_DIR / cls
    class_dir.mkdir(exist_ok=True)

    positives = cfg["positives"]
    uncond_prompt = cfg["negative"]

    print(f"\nGenerating images for class: {cls}")

    for i in range(NUM_IMAGES_PER_CLASS):
        semantic_clause = random.choice(positives)

        prompt = (
            f"A realistic photograph of a {cls}. "
            f"{semantic_clause}. "
            "Single primary subject, natural environment. "
            "Captured with real-world lighting and perspective."
        )

        seed = random.randint(0, 2**32 - 1)

        output_image = pipeline.generate(
            prompt=prompt,
            uncond_prompt=uncond_prompt,
            input_image=None,
            strength=1.0,
            do_cfg=do_cfg,
            cfg_scale=cfg_scale,
            sampler_name=sampler,
            n_inference_steps=num_inference_steps,
            seed=seed,
            models=models,
            device=DEVICE,
            idle_device="cpu",
            tokenizer=tokenizer,
        )

        image = Image.fromarray(output_image)
        image.save(class_dir / f"{cls}_{i:03d}.png")

        print(f"  Saved {cls}_{i:03d}.png")

print("\nSemantic synthetic dataset generation with CFG complete.")

/var/folders/4v/29svpk3155lcw7lcnd3b1dsr0000gn/T/ipykernel_52801/243064508.py:78: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  elif (torch.has_mps or torch.backends.mps.is_available()) and ALLOW_MPS:


Using device: mps

Generating images for class: tiger


100%|██████████| 100/100 [00:53<00:00,  1.85it/s]


  Saved tiger_000.png


100%|██████████| 100/100 [00:55<00:00,  1.82it/s]


...

  Saved tiger_003.png


100%|██████████| 100/100 [00:54<00:00,  1.83it/s]


  Saved tiger_004.png


 11%|█         | 11/100 [00:06<00:51,  1.73it/s]


KeyboardInterrupt: 